# Minimum Teacher Investment + DCT Sweep

**Two questions in one eval:**
1. How little teacher training produces useful spectra?
2. How few DCT coefficients (bytes) are needed?

**Design:**
- Contiguous Train/Val/Test split (preserves sequential structure)
- Teacher sweep: {50, 100, 200, 500, 1000, 2000} steps
- DCT sweep: {2, 4, 8, 16} coefficients
- Students use spectral-only Marathon (no directions = pure Tier 1)

**Output:** Teacher steps × DCT coefficients → student speedup matrix

In [ ]:
# Cell 1: Setup
import os, subprocess, shutil

# Clean start
os.chdir('/content')
if os.path.exists('/content/nanogpt-prism'):
    shutil.rmtree('/content/nanogpt-prism')

subprocess.run(['git', 'clone', 'https://github.com/timepointai/nanogpt-prism-shakespeare.git',
                '/content/nanogpt-prism'], check=True)
os.chdir('/content/nanogpt-prism')
print(f'Working dir: {os.getcwd()}')

subprocess.run(['pip', 'install', '-q', 'transformers', 'tiktoken', 'datasets'])

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

subprocess.run(['python', 'data/shakespeare_char/prepare.py'],
               capture_output=True)
assert os.path.exists('data/shakespeare_char/train.bin'), 'prepare.py failed'
print('Shakespeare dataset ready.')

In [ ]:
# Cell 2: Create contiguous data partition
import numpy as np, os, shutil
os.chdir('/content/nanogpt-prism')

train_all = np.array(np.memmap('data/shakespeare_char/train.bin', dtype=np.uint16, mode='r'))
test_data = np.array(np.memmap('data/shakespeare_char/val.bin', dtype=np.uint16, mode='r'))

split = int(len(train_all) * 0.80)
train_data = train_all[:split].astype(np.uint16)
teacher_val = train_all[split:].astype(np.uint16)

for name, train, val in [('shakespeare_eval', train_data, test_data),
                          ('shakespeare_teacher', train_data, teacher_val)]:
    d = f'data/{name}'
    os.makedirs(d, exist_ok=True)
    train.tofile(os.path.join(d, 'train.bin'))
    val.tofile(os.path.join(d, 'val.bin'))
    shutil.copy('data/shakespeare_char/meta.pkl', os.path.join(d, 'meta.pkl'))

print(f'Train:       {len(train_data):>10,} tokens')
print(f'Teacher-Val: {len(teacher_val):>10,} tokens')
print(f'Test:        {len(test_data):>10,} tokens (held out)')
print('Contiguous splits ready.')

In [ ]:
# Cell 3: Train teachers + extract spectra at multiple n_dct
import subprocess, os, re
os.chdir('/content/nanogpt-prism')

teacher_steps_list = [50, 100, 200, 500, 1000, 2000]
n_dct_list = [2, 4, 8, 16]

for steps in teacher_steps_list:
    ckpt = f'out-teacher-{steps}/ckpt.pt'
    if os.path.exists(ckpt):
        print(f'  Teacher {steps}: exists.')
        continue
    print(f'  Training teacher ({steps} steps)...')
    result = subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        '--dataset=shakespeare_teacher',
        f'--max_iters={steps}', f'--eval_interval={steps}',
        '--eval_iters=50', f'--log_interval={steps}',
        f'--out_dir=out-teacher-{steps}',
        '--always_save_checkpoint=True', '--compile=False',
        '--prism_init=False', '--wandb_log=False',
    ], capture_output=True, text=True, timeout=600)
    for line in result.stdout.split('\n'):
        m = re.search(r'step \d+: train loss ([\d.]+), val loss ([\d.]+)', line)
        if m: print(f'    val: {m.group(2)}')
    if result.returncode != 0:
        print(f'    ERR: {result.stderr[-200:]}')

for steps in teacher_steps_list:
    for n_dct in n_dct_list:
        cache = f'.prism_cache/t{steps}_d{n_dct}'
        if os.path.exists(f'{cache}/spectra.json'):
            continue
        subprocess.run([
            'python', 'prism_extract.py',
            '--ckpt', f'out-teacher-{steps}/ckpt.pt',
            '--out', cache, f'--n_dct={n_dct}',
        ], capture_output=True, timeout=120)
        raw_bytes = n_dct * 5 * 4
        print(f'  t{steps}_d{n_dct}: {raw_bytes} bytes')

print('\nAll extracted.')

In [ ]:
# Cell 4: Run all students
import subprocess, time, re, json, os
os.chdir('/content/nanogpt-prism')

STEPS = 5000
EVAL = 100

configs = [('baseline', ['--prism_init=False'])]
for ts in [50, 100, 200, 500, 1000, 2000]:
    for nd in [2, 4, 8, 16]:
        configs.append((f't{ts}_d{nd}', [
            '--prism_init=True', '--prism_align=0.0',
            f'--prism_spectra=.prism_cache/t{ts}_d{nd}/spectra.json',
            '--learning_rate=5e-4', '--warmup_iters=50',
            '--prism_mod=0.01', '--prism_mod_decay=0.9999',
        ]))
configs.append(('full_t2000', [
    '--prism_init=True', '--prism_align=0.75',
    '--prism_spectra=.prism_cache/t2000_d8/spectra.json',
    '--prism_directions=.prism_cache/t2000_d8/directions.pt',
    '--learning_rate=5e-4', '--warmup_iters=50',
    '--prism_mod=0.01', '--prism_mod_decay=0.9999',
]))

all_curves = {}
for name, extra in configs:
    log_path = f'/content/mt_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'  {name}...')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             '--dataset=shakespeare_eval',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=500',
             f'--out_dir=out-mt-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'    ERR: {result.stderr[-200:]}')
        print(f'    {time.time()-t0:.0f}s')
    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m: val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    if val: print(f'  [{name}] best: {min(val.values()):.4f}')

with open('/content/mt_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 5: Results matrix
import json

curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/mt_curves.json')).items()}

bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*75)
print('  TEACHER STEPS x DCT COEFFICIENTS -> STUDENT SPEEDUP')
print('='*75)
print(f'  Contiguous partition. Eval on held-out test set.')
print(f'  Baseline best: {bb:.4f} at step {bs}')

n_groups = 5
print(f'\n  --- Speedup to reach baseline best ---')
header = f'{"":>12s}'
for nd in [2, 4, 8, 16]:
    b = nd * n_groups * 4
    header += f'  d={nd} ({b}B)'.rjust(14)
print(f'  {header}')
print(f'  {"-"*72}')
for ts in [50, 100, 200, 500, 1000, 2000]:
    row = f'{ts} steps'.rjust(12)
    for nd in [2, 4, 8, 16]:
        c = curves.get(f't{ts}_d{nd}', {})
        if not c:
            row += '—'.rjust(14)
            continue
        hit = next((s for s in sorted(c) if c[s] <= bb), None)
        if hit:
            row += f'{bs/hit:.1f}x'.rjust(14)
        else:
            row += f'({min(c.values()):.3f})'.rjust(14)
    print(f'  {row}')

c = curves.get('full_t2000', {})
if c:
    hit = next((s for s in sorted(c) if c[s] <= bb), None)
    spd = f'{bs/hit:.1f}x' if hit else '—'
    print(f'\n  Full Tier 2 (2000 steps + directions): {spd}')

print(f'\n  --- Best val loss (test set) ---')
header = f'{"":>12s}'
for nd in [2, 4, 8, 16]:
    header += f'd={nd}'.rjust(14)
print(f'  {header}')
print(f'  {"-"*72}')
for ts in [50, 100, 200, 500, 1000, 2000]:
    row = f'{ts} steps'.rjust(12)
    for nd in [2, 4, 8, 16]:
        c = curves.get(f't{ts}_d{nd}', {})
        if c:
            best = min(c.values())
            m = ' *' if best < bb else '  '
            row += f'{best:>12.4f}{m}'
        else:
            row += '—'.rjust(14)
    print(f'  {row}')
print(f'  {"Baseline":>12s}  {bb:.4f}')
print(f'\n  * = beats baseline. () = never reached baseline.')

In [ ]:
# Cell 6: Download
from google.colab import files
files.download('/content/mt_curves.json')